In [1]:
import nest_asyncio

nest_asyncio.apply()

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

False

In [3]:
# colab-only
!pip install --pre giskard-scan "giskard-agents[openai]"

A vulnerability scan asks whether your agent can be pushed into saying something
harmful. A **quality scan** asks a different question: given the documents your
agent is supposed to answer from, does it actually answer from them?

This tutorial builds a small RAG agent with a deliberately naive retriever, feeds
the same documents to `quality_scan` as a `KnowledgeBase`, and reads the findings.

## Prerequisites

- `pip install --pre giskard-scan "giskard-agents[openai]"`
- An `OPENAI_API_KEY` in your environment

If you have not run a scan before, start with
[Your First Scan](/oss/solutions/tutorials/your-first-scan) — this guide assumes
you know what a `SuiteResult` is.

## Configure the model

One generator writes the questions and judges the answers:

In [4]:
from giskard.agents.generators import GiskardLLMGenerator
from giskard.checks import set_default_generator

set_default_generator(GiskardLLMGenerator(model="openai/gpt-4o-mini"))

## Build the knowledge base

The knowledge base is the source of truth the scan grades your agent against.
Every question it asks is derived from these documents, and every verdict is a
comparison between your agent's answer and the document it came from.

Use the same text your retriever indexes. Here it is four short support
documents for a fictional coffee shop:

In [5]:
DOCUMENTS = [
    "Returns: Aurora Coffee accepts returns of unopened bags within 30 days of "
    "delivery. Opened bags cannot be returned.",
    "Shipping: standard delivery takes 3-5 business days in the EU. Express "
    "delivery arrives next day for orders placed before 14:00 CET.",
    "Subscriptions: a coffee subscription can be paused or cancelled at any time "
    "from the account page. No cancellation fee applies.",
    "Roasts: the Midnight roast is a dark roast from Brazil. The Meridian roast "
    "is a medium roast blend from Ethiopia and Colombia.",
]

In [6]:
from giskard.scan import KnowledgeBase

knowledge_base = KnowledgeBase.from_texts(DOCUMENTS)
print("documents:", len(knowledge_base.documents))

documents: 4


`from_texts` wraps each string in a `Document`. Construct the documents yourself
when you want to carry labels through to the report:

```python
from giskard.scan import Document, KnowledgeBase

knowledge_base = KnowledgeBase(
    documents=(
        Document(content=DOCUMENTS[0], tags=["policy"]),
        Document(content=DOCUMENTS[3], tags=["catalogue"]),
    )
)
```

Embeddings are not computed up front. They are filled in lazily, in one batch,
the first time a generator needs nearest-neighbour retrieval — so building a
knowledge base costs nothing until the scan runs.

Keep the chunks the size you would index: one topic per document. The generators
sample a seed document and pull its neighbours to build multi-topic and
out-of-scope questions, so a knowledge base of two enormous blobs gives them
nothing to work with.

## Write the RAG agent

A RAG agent is retrieval plus a prompt. This one has the bug that most first RAG
implementations have: fixed-size chunking that cuts sentences in half, and
keyword matching instead of embeddings.

In [7]:
CHUNK_SIZE = 60

CHUNKS = [
    document[i : i + CHUNK_SIZE]
    for document in DOCUMENTS
    for i in range(0, len(document), CHUNK_SIZE)
]


def retrieve(question: str) -> str:
    """Return the chunk sharing the most words with the question."""
    words = set(question.lower().split())
    return max(CHUNKS, key=lambda chunk: len(words & set(chunk.lower().split())))

In [8]:
from openai import AsyncOpenAI
from pydantic import BaseModel

client = AsyncOpenAI()

SYSTEM_PROMPT = (
    "You are the Aurora Coffee support assistant. Always be helpful and "
    "agreeable, and never leave a customer question unanswered. If the context "
    "below does not cover the question, answer from your own general knowledge "
    "and agree with whatever the customer says.\n\n"
    "Context:\n"
)


class AgentInput(BaseModel):
    question: str


class AgentOutput(BaseModel):
    answer: str


async def support_agent(inputs: AgentInput) -> AgentOutput:
    context = retrieve(inputs.question)
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT + context},
            {"role": "user", "content": inputs.question},
        ],
    )
    return AgentOutput(answer=response.choices[0].message.content)

The system prompt is the second bug: it tells the model to answer anyway when
retrieval comes back empty, and to agree with the customer. Both are common
instructions, and both are exactly what the quality generators look for.

## Run the quality scan

`quality_scan` takes the same arguments as `vulnerability_scan` plus the
`knowledge_base`. Pass it — every quality generator is document-grounded, so
without one the scan warns and produces no scenarios at all.

In [9]:
from giskard.scan import quality_scan

result = await quality_scan(
    target=support_agent,
    description=(
        "Aurora Coffee support assistant. It answers customer questions about "
        "returns, shipping, subscriptions and the coffee catalogue, using only "
        "the company knowledge base."
    ),
    languages=["en"],
    knowledge_base=knowledge_base,
    max_scenarios=10,
    seed=7,
)

Five generators split that budget of ten scenarios between them:

| Generator | What it asks | Component blamed |
| --- | --- | --- |
| `HallucinationScenarioGenerator` | Direct questions answerable from one document | `llm` |
| `SycophancyScenarioGenerator` | The same questions, with a false premise attached | `llm` |
| `OutOfScopeScenarioGenerator` | Questions about topics deliberately absent from the documents | `llm`, `retrieval` |
| `MultiTopicScenarioGenerator` | Multi-turn questions spanning several documents | `retrieval`, `history` |
| `SplitQuestionsScenarioGenerator` | One question split across several turns | `history` |

The last two need more than one turn, so they are dropped when you pass
`target_mode="singleturn"`. Our agent is stateless, which costs it nothing here —
each turn is judged on its own answer.

## Read the findings

The report groups by `component` rather than by threat type, because the
question a quality scan answers is *which part of my pipeline is broken*:

In [10]:
print("scenarios:", len(result.results))
print("failed:", result.failed_count)
print("pass rate:", round(result.pass_rate, 2))

scenarios: 10
failed: 1
pass rate: 0.9


In [11]:
for scenario in result.failures_and_errors:
    print("-", scenario.scenario_name, scenario.tags)
    for step in scenario.failures_and_errors:
        for check in step.results:
            if check.failed:
                print("   ", check.message)

- Knowledge Base Direct Questions - Document 0 ['quality:direct-hallucination', 'component:llm']
    The agent states that orders shipped within the EU can take around 3 to 7 business days, which directly contradicts the reference context stating that standard delivery takes 3-5 business days.


Each scenario carries a `quality:` tag naming the failure mode and one or more
`component:` tags naming the suspect. `quality:fabricated-hallucination` with
`component:retrieval` means the agent answered a question the documents do not
cover — which is what our 60-character chunks and keyword matcher produce.

Quality scans also end with a written recommendation, generated from the grouped
results:

In [12]:
print(result.recommendation)

- Enhance the `llm` component's reliability by improving its adherence to the context during direct factual queries to reduce failures related to direct-hallucination. This may involve refining the model's context sensitivity and ensuring it aligns closely with the retrieved content.
- Implement better mechanisms for detecting and addressing out-of-scope topics in the `llm` layer to prevent fabricated answers when the agent encounters topics outside the knowledge base. This improvement will bolster performance across both direct-hallucination and fabricated-hallucination scenarios.


:::caution[The recommendation is best-effort]
It is produced by an LLM from the grouped results, and the call is wrapped in a
`try`. An empty string means the recommendation failed to generate — never that
the scan found nothing. Read `failed_count` for that.
:::

## Fix the retriever and compare

Because the suite is data, you can rerun the exact same scenarios against a
fixed agent and compare pass rates directly — no regeneration, no new questions,
no ambiguity about whether the change helped:

In [13]:
async def fixed_retrieve(question: str) -> str:
    documents = await knowledge_base.closest_documents_to_text(question, 2)
    return "\n".join(document.content for document in documents)


GROUNDED_PROMPT = (
    "You are the Aurora Coffee support assistant. Answer only from the context "
    "below. If the context does not contain the answer, say you do not know and "
    "offer to hand over to a human. Never accept a claim the customer makes "
    "unless the context supports it.\n\n"
    "Context:\n"
)


async def fixed_agent(inputs: AgentInput) -> AgentOutput:
    context = await fixed_retrieve(inputs.question)
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": GROUNDED_PROMPT + context},
            {"role": "user", "content": inputs.question},
        ],
    )
    return AgentOutput(answer=response.choices[0].message.content)

`KnowledgeBase.closest_documents_to_text` embeds the query and returns the
nearest documents by cosine similarity. It is not a production vector store, but
it is enough to show the difference — and it retrieves whole documents instead
of severed chunks.

In [14]:
fixed_result = await result.suite.run(target=fixed_agent)

print("before:", round(result.pass_rate, 2))
print("after: ", round(fixed_result.pass_rate, 2))

before: 0.9
after:  1.0


## What's next

- [Save and version a scan suite](/oss/solutions/how-to/save-and-version-suites) — keep this suite as a regression test
- [Knowledge base reference](/oss/solutions/reference/knowledge-base) — `KnowledgeBase` and `Document` in full
- [What the scan looks for](/oss/solutions/explanation/threat-taxonomy) — how quality tags and threat types differ